CLIP

In [ ]:
!unzip csv_style.zip

Archive:  csv_style.zip
   creating: csv_style/
  inflating: csv_style/test_style.csv  
  inflating: csv_style/train_style.csv  
  inflating: csv_style/val_style.csv  


In [ ]:
!unzip all_images.zip

Archive:  all_images.zip
   creating: all_images/
  inflating: all_images/001_beaee5fe.jpg  
  inflating: all_images/001_d2c7428a.jpg  
  inflating: all_images/002_5ef9df19.jpg  
  inflating: all_images/002_8cdb5424.jpg  
  inflating: all_images/002_a45d2d3c.jpg  
  inflating: all_images/003_4600a7ff.jpg  
  inflating: all_images/003_bb4ea859.jpg  
  inflating: all_images/003_cfd2b5f1.jpg  
  inflating: all_images/004_7a46c013.jpg  
  inflating: all_images/004_c3667694.jpg  
  inflating: all_images/004_cabd9dfd.jpg  
  inflating: all_images/005_86125b98.jpg  
  inflating: all_images/005_dbecd9c4.jpg  
  inflating: all_images/005_eda52226.jpg  
  inflating: all_images/006_221a0ab3.jpg  
  inflating: all_images/006_6fd00887.jpg  
  inflating: all_images/006_ec7e3943.jpg  
  inflating: all_images/007_64a3e2e9.jpg  
  inflating: all_images/007_97626a67.jpg  
  inflating: all_images/007_f2954e4e.jpg  
  inflating: all_images/008_52baff74.jpg  
  inflating: all_images/008_88b44079.jpg  
  in

In [ ]:
# !pip install -q -U transformers torch pillow pandas scikit-learn tqdm

from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

import os
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_DIR = "/content/all_images"
MODEL_NAME = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

def to_tensor(out):
    return out.pooler_output if hasattr(out, "pooler_output") else out

# ---------- Классы и промпты ----------
CLASS_NAMES = ["rustic", "farmhouse", "modern"]
CLASS_PROMPTS = [f"a photo of a {c} style house" for c in CLASS_NAMES]

@torch.no_grad()
def encode_texts(texts):
    inputs = processor(text=texts, return_tensors="pt",
                       padding=True, truncation=True).to(DEVICE)
    emb = to_tensor(model.get_text_features(**inputs))
    return emb / emb.norm(dim=-1, keepdim=True)

@torch.no_grad()
def encode_image(path):
    img = Image.open(path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)
    emb = to_tensor(model.get_image_features(**inputs))
    return emb / emb.norm(dim=-1, keepdim=True)

class_embeds = encode_texts(CLASS_PROMPTS)   # (3, D)

# ---------- Прогон по CSV ----------
def run_split(csv_path, alpha=0.5):
    df = pd.read_csv(csv_path)
    y_true = []
    y_img, y_txt, y_mul = [], [], []
    rows = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=os.path.basename(csv_path)):
        path = os.path.join(IMG_DIR, row["filename"])
        try:
            img_emb = encode_image(path)
        except FileNotFoundError:
            continue

        txt_emb = encode_texts([str(row["text"])])

        p_img = (img_emb @ class_embeds.T).softmax(dim=-1).squeeze(0)
        p_txt = (txt_emb @ class_embeds.T).softmax(dim=-1).squeeze(0)

        combined = alpha * img_emb + (1 - alpha) * txt_emb
        combined = combined / combined.norm(dim=-1, keepdim=True)
        p_mul = (combined @ class_embeds.T).softmax(dim=-1).squeeze(0)

        pred_img = CLASS_NAMES[p_img.argmax().item()]
        pred_txt = CLASS_NAMES[p_txt.argmax().item()]
        pred_mul = CLASS_NAMES[p_mul.argmax().item()]

        y_true.append(row["label"])
        y_img.append(pred_img)
        y_txt.append(pred_txt)
        y_mul.append(pred_mul)

        rows.append({
            "filename": row["filename"],
            "true": row["label"],
            "img_only": pred_img,
            "text_only": pred_txt,
            "multimodal": pred_mul,
        })

    return pd.DataFrame(rows), y_true, y_img, y_txt, y_mul

# ---------- Метрики ----------
def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"\n=== {name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Macro-F1:  {f1:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("Confusion matrix (rows=true, cols=pred, order:", CLASS_NAMES, ")")
    print(confusion_matrix(y_true, y_pred, labels=CLASS_NAMES))
    return acc, f1

# ---------- Основной прогон ----------
results = {}
for split_name, csv_path in [
    ("train", "/content/csv_style/train_style.csv"),
    ("val",   "/content/csv_style/val_style.csv"),
    ("test",  "/content/csv_style/test_style.csv"),
]:
    preds_df, yt, yi, ytx, ym = run_split(csv_path)
    preds_df.to_csv(f"/content/preds_{split_name}.csv", index=False)
    results[split_name] = {
        "n": len(yt),
        "image_only": report(f"{split_name} — image-only", yt, yi),
        "text_only":  report(f"{split_name} — text-only",  yt, ytx),
        "multimodal": report(f"{split_name} — multimodal", yt, ym),
    }

# ---------- Сводка ----------
summary = []
for split, r in results.items():
    for mode in ["image_only", "text_only", "multimodal"]:
        acc, f1 = r[mode]
        summary.append({"split": split, "mode": mode, "n": r["n"],
                        "accuracy": round(acc, 4), "macro_f1": round(f1, 4)})
summary_df = pd.DataFrame(summary)
print("\n=== Сводка по всем сплитам ===")
print(summary_df.pivot(index="split", columns="mode",
                       values=["accuracy", "macro_f1"]))
summary_df.to_csv("/content/summary.csv", index=False)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

train_style.csv:   0%|          | 0/26 [00:00<?, ?it/s]


=== train — image-only ===
Accuracy:  0.7692
Macro-F1:  0.7202
              precision    recall  f1-score   support

   farmhouse       0.73      0.85      0.79        13
      modern       0.78      1.00      0.88         7
      rustic       1.00      0.33      0.50         6

    accuracy                           0.77        26
   macro avg       0.84      0.73      0.72        26
weighted avg       0.81      0.77      0.74        26

Confusion matrix (rows=true, cols=pred, order: ['rustic', 'farmhouse', 'modern'] )
[[ 2  4  0]
 [ 0 11  2]
 [ 0  0  7]]

=== train — text-only ===
Accuracy:  0.4231
Macro-F1:  0.3810
              precision    recall  f1-score   support

   farmhouse       0.00      0.00      0.00        13
      modern       0.55      0.86      0.67         7
      rustic       0.33      0.83      0.48         6

    accuracy                           0.42        26
   macro avg       0.29      0.56      0.38        26
weighted avg       0.22      0.42      0.29   

val_style.csv:   0%|          | 0/100 [00:00<?, ?it/s]


=== val — image-only ===
Accuracy:  0.8200
Macro-F1:  0.7976
              precision    recall  f1-score   support

   farmhouse       0.87      0.84      0.85        49
      modern       0.72      1.00      0.84        28
      rustic       0.93      0.57      0.70        23

    accuracy                           0.82       100
   macro avg       0.84      0.80      0.80       100
weighted avg       0.84      0.82      0.81       100

Confusion matrix (rows=true, cols=pred, order: ['rustic', 'farmhouse', 'modern'] )
[[13  6  4]
 [ 1 41  7]
 [ 0  0 28]]

=== val — text-only ===
Accuracy:  0.5300
Macro-F1:  0.5087
              precision    recall  f1-score   support

   farmhouse       0.88      0.14      0.25        49
      modern       0.59      0.96      0.73        28
      rustic       0.41      0.83      0.55        23

    accuracy                           0.53       100
   macro avg       0.62      0.64      0.51       100
weighted avg       0.69      0.53      0.45       

test_style.csv:   0%|          | 0/401 [00:00<?, ?it/s]


=== test — image-only ===
Accuracy:  0.8254
Macro-F1:  0.8219
              precision    recall  f1-score   support

   farmhouse       0.85      0.79      0.82       195
      modern       0.74      0.99      0.85       113
      rustic       0.94      0.69      0.80        93

    accuracy                           0.83       401
   macro avg       0.84      0.82      0.82       401
weighted avg       0.84      0.83      0.82       401

Confusion matrix (rows=true, cols=pred, order: ['rustic', 'farmhouse', 'modern'] )
[[ 64  27   2]
 [  3 155  37]
 [  1   0 112]]

=== test — text-only ===
Accuracy:  0.5636
Macro-F1:  0.5409
              precision    recall  f1-score   support

   farmhouse       0.97      0.14      0.25       195
      modern       0.63      0.96      0.76       113
      rustic       0.45      0.97      0.61        93

    accuracy                           0.56       401
   macro avg       0.68      0.69      0.54       401
weighted avg       0.75      0.56      

Генерация описаний

In [ ]:
# !pip install -q -U transformers torch pillow pandas tqdm accelerate

from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

import os
import json
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText

# ============================================================
# 1. Загрузка модели Qwen3-VL
# ============================================================
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"   # 2B влезает в T4 без квантизации
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

# ============================================================
# 2. Промпт для генерации стилевого описания
# ============================================================
STYLE_PROMPT = (
    """Look at this house. Describe its architectural features in ONE short sentence, max 15 words.
Focus ONLY on: building materials (wood, stone, brick, glass, siding, concrete),
roof shape (gabled, flat, pitched, hipped), and structural elements
(porch, exposed beams, large windows, chimney, minimalist geometric forms).
Do NOT use the words 'rustic', 'farmhouse', 'modern', or 'style'.
Do NOT mention colors, driveways, lawns, cars, or generic objects."""
)

def generate_description(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return None

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": STYLE_PROMPT},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=30, do_sample=False)

    # отрезаем промпт от ответа
    gen = out[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()

    # чистим: убираем переносы, кавычки, "The house..."
    text = text.replace("\n", " ").strip().strip('"').strip("'")
    if text.lower().startswith("the house "):
        text = "a house " + text[len("the house "):]
    return text

# ============================================================
# 3. Проход по всем изображениям
# ============================================================
IMG_DIR = "/content/all_images"
CACHE_PATH = "/content/style_descriptions.json"

# подхватываем кэш, чтобы не генерить заново при перезапуске
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        descriptions = json.load(f)
    print(f"Загружено из кэша: {len(descriptions)} описаний")
else:
    descriptions = {}

image_files = sorted([
    f for f in os.listdir(IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
])
print(f"Всего изображений: {len(image_files)}")

for fname in tqdm(image_files, desc="Генерация описаний"):
    if fname in descriptions:
        continue
    path = os.path.join(IMG_DIR, fname)
    desc = generate_description(path)
    descriptions[fname] = desc if desc else ""

    # сохраняем кэш каждые 50 изображений
    if len(descriptions) % 50 == 0:
        with open(CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(descriptions, f, ensure_ascii=False, indent=2)

# финальное сохранение кэша
with open(CACHE_PATH, "w", encoding="utf-8") as f:
    json.dump(descriptions, f, ensure_ascii=False, indent=2)
print(f"Готово. Сгенерировано описаний: {len(descriptions)}")

# ============================================================
# 4. Замена колонки text в CSV
# ============================================================
SPLITS = {
    "train": "/content/train.csv",
    "val":   "/content/val.csv",
    "test":  "/content/test.csv",
}
OUT_DIR = "/content/csv_style"
os.makedirs(OUT_DIR, exist_ok=True)

for name, path in SPLITS.items():
    df = pd.read_csv(path)
    old_texts = df["text"].copy()

    # заменяем text на новое описание из словаря
    df["text"] = df["filename"].map(descriptions)

    # диагностика: сколько не нашлось и сколько пустых
    missing = df["text"].isna().sum()
    empty = (df["text"].astype(str).str.strip() == "").sum()

    # для отсутствующих — оставляем старое описание
    df.loc[df["text"].isna() | (df["text"].astype(str).str.strip() == ""), "text"] = old_texts

    out_path = os.path.join(OUT_DIR, f"{name}_style.csv")
    df.to_csv(out_path, index=False)

    print(f"\n{name}: {len(df)} строк")
    print(f"  пропущено (нет в папке): {missing}")
    print(f"  пустых ответов модели:   {empty}")
    print(f"  сохранено: {out_path}")
    print(f"  примеры новых описаний:")
    for _, row in df.head(3).iterrows():
        print(f"    [{row['label']}] {row['text']}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Всего изображений: 867


Генерация описаний:   0%|          | 0/867 [00:00<?, ?it/s]

Готово. Сгенерировано описаний: 867

train: 26 строк
  пропущено (нет в папке): 0
  пустых ответов модели:   0
  сохранено: /content/csv_style/train_style.csv
  примеры новых описаний:
    [modern] A two-story house with wood siding, concrete foundations, glass railings, a gabled roof, large windows, and minimalist geometric forms.
    [modern] A two-story house with stone cladding, glass windows, a flat roof, and minimalist geometric forms featuring a balcony and exposed beams.
    [rustic] A wooden, stone, and glass house with a gabled roof, exposed beams, large windows, a chimney, and a porch.

val: 100 строк
  пропущено (нет в папке): 0
  пустых ответов модели:   0
  сохранено: /content/csv_style/val_style.csv
  примеры новых описаний:
    [modern] Concrete and stone walls, glass balcony, minimalist geometric forms, large windows, and a flat roof.
    [rustic] Wooden beams, stone pillars, and a gabled roof with a porch and exposed beams.
    [modern] A multi-story house with stone 

Классификация

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# !pip install -q -U transformers accelerate pillow pandas torch

from google.colab import userdata
import torch
import pandas as pd
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# ---------- Загрузка модели ----------
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

IMG_DIR = "/content/all_images"
CLASS_NAMES = ["rustic", "farmhouse", "modern"]
QUESTION = "Classify the style of this house. Answer with one word only: rustic, farmhouse, or modern."

# ---------- Загрузка CSV ----------
train_df = pd.read_csv("/content/csv_style/train_style.csv")
test_df = pd.read_csv("/content/csv_style/test_style.csv")

# ---------- Формирование few-shot примеров (по 1 на класс) ----------
few_shot_examples = []  # [(path, label), ...]
for cls in CLASS_NAMES:
    row = train_df[train_df["label"] == cls].iloc[0]
    few_shot_examples.append((f"{IMG_DIR}/{row['filename']}", cls))
    print(f"Few-shot пример [{cls}]: {row['filename']}")

# ---------- Тестовые примеры (по 1 на класс) ----------
test_examples = []  # [(path, true_label), ...]
for cls in CLASS_NAMES:
    row = test_df[test_df["label"] == cls].iloc[0]
    test_examples.append((f"{IMG_DIR}/{row['filename']}", cls))
    print(f"Тестовый пример [{cls}]: {row['filename']}")

# ---------- Функция few-shot классификации ----------
def classify_few_shot(test_image_path):
    messages = []

    # Добавляем few-shot примеры как пары user/assistant
    for img_path, label in few_shot_examples:
        ex_img = Image.open(img_path).convert("RGB")
        messages.append({
            "role": "user",
            "content": [
                {"type": "image", "image": ex_img},
                {"type": "text", "text": QUESTION},
            ],
        })
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": label}],
        })

    # Тестовое изображение
    test_img = Image.open(test_image_path).convert("RGB")
    messages.append({
        "role": "user",
        "content": [
            {"type": "image", "image": test_img},
            {"type": "text", "text": QUESTION},
        ],
    })

    # Прогон через процессор и генерация
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)

    # Декодируем только сгенерированную часть
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    answer = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip().lower()

    # Парсим ответ: ищем первое совпадение с классом
    for c in CLASS_NAMES:
        if c in answer:
            return c
    return answer  # если ничего не нашли — вернём как есть, чтобы увидеть

# ---------- Прогон по 3 тестовым примерам ----------
print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ FEW-SHOT КЛАССИФИКАЦИИ (3 примера в контексте)")
print("=" * 60)

correct = 0
for test_path, true_label in test_examples:
    pred = classify_few_shot(test_path)
    is_correct = (pred == true_label)
    correct += int(is_correct)
    mark = "✓" if is_correct else "✗"
    print(f"{mark} файл: {test_path.split('/')[-1]}")
    print(f"   истина:   {true_label}")
    print(f"   предсказ: {pred}")
    print()

print("=" * 60)
print(f"Accuracy: {correct}/{len(test_examples)} = {correct / len(test_examples):.2%}")

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Few-shot пример [rustic]: 187_f9d3cf18.jpg
Few-shot пример [farmhouse]: 007_97626a67.jpg
Few-shot пример [modern]: 006_221a0ab3.jpg
Тестовый пример [rustic]: 042_06b56791.jpg
Тестовый пример [farmhouse]: 243_6a3aad07.jpg
Тестовый пример [modern]: 054_37cbf3d3.jpg

РЕЗУЛЬТАТЫ FEW-SHOT КЛАССИФИКАЦИИ (3 примера в контексте)
✗ файл: 042_06b56791.jpg
   истина:   rustic
   предсказ: farmhouse

✓ файл: 243_6a3aad07.jpg
   истина:   farmhouse
   предсказ: farmhouse

✓ файл: 054_37cbf3d3.jpg
   истина:   modern
   предсказ: modern

Accuracy: 2/3 = 66.67%


In [ ]:
# !pip install -q openai pillow
from google.colab import userdata

from openai import OpenAI
import base64
from io import BytesIO
from PIL import Image

client = OpenAI(
    api_key=userdata.get('ROUTERAI_KEY'),   # ключ из личного кабинета RouterAI
    base_url="https://routerai.ru/api/v1",
)

# --- Тест 1: простой текстовый запрос (без картинки) ---
try:
    resp = client.chat.completions.create(
        model="qwen/qwen3.8-omni-flash",
        messages=[{"role": "user", "content": "Say hi in one word."}],
        max_tokens=10,
    )
    print("Текстовый тест OK:", resp.choices[0].message.content)
    print("Использовано токенов:", resp.usage.total_tokens)
except Exception as e:
    print("Ошибка:", type(e).__name__, e)

Текстовый тест OK: Hi
Использовано токенов: 89


In [ ]:
import base64
from io import BytesIO
from PIL import Image

def encode_image(path, max_size=512):
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size))
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

b64 = encode_image("/content/all_images/243_6a3aad07.jpg")

try:
    resp = client.chat.completions.create(
        model="openai/gpt-6-luna",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "Classify the style of this house."},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                {"type": "text",
                 "text": "Answer with one word only: rustic, farmhouse, or modern."},
            ],
        }],
        max_tokens=20,
        temperature=0,
        extra_body={"reasoning": {"enabled": False}},
        reasoning_effort="none"
    )
    print("Ответ:", resp.choices[0].message.content)
    print("Всего токенов:", resp.usage.total_tokens)
    print("Входных токенов:", resp.usage.prompt_tokens)
    print("Выходных токенов:", resp.usage.completion_tokens)

except Exception as e:
    print("Ошибка:", type(e).__name__, e)

Ответ: farmhouse
Всего токенов: 283
Входных токенов: 277
Выходных токенов: 6


In [ ]:
resp

ChatCompletion(id='rai-1791337537-Ozq34aavgOUoEdnpBWdQ', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need answer one word only. Need classify style. Image: white board-and-batten? g'))], created=1791337537, model='deepseek-flash', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='aeb56401ca74e127821c4f9126dcb669', usage=CompletionUsage(completion_tokens=20, prompt_tokens=250, total_tokens=270, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=20, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0), cost=0.012134191212000002), provider='DeepSeek')